In [ ]:
import zipfile

with zipfile.ZipFile("/content/clean_dataset.zip", 'r') as z:
    z.extractall("/content/malnutrition_raw")

print("DONE EXTRACTING")

DONE EXTRACTING


In [ ]:
!find /content/malnutrition_raw -maxdepth 3

/content/malnutrition_raw
/content/malnutrition_raw/content
/content/malnutrition_raw/content/clean_dataset
/content/malnutrition_raw/content/clean_dataset/dataFinal


In [ ]:
import zipfile

with zipfile.ZipFile("/content/FGNET.zip", 'r') as z:
    z.extractall("/content/FGNET")

print("FGNET extracted")

FGNET extracted


In [ ]:
!find /content/FGNET -maxdepth 3

/content/FGNET
/content/FGNET/__MACOSX
/content/FGNET/__MACOSX/FGNET
/content/FGNET/__MACOSX/FGNET/._.DS_Store
/content/FGNET/__MACOSX/FGNET/points
/content/FGNET/FGNET
/content/FGNET/FGNET/images
/content/FGNET/FGNET/images/080A04.JPG
/content/FGNET/FGNET/images/003A35.JPG
/content/FGNET/FGNET/images/056A06.JPG
/content/FGNET/FGNET/images/040A14.JPG
/content/FGNET/FGNET/images/041A04.JPG
/content/FGNET/FGNET/images/011A05.JPG
/content/FGNET/FGNET/images/013A19.JPG
/content/FGNET/FGNET/images/068A10a.JPG
/content/FGNET/FGNET/images/071A18.JPG
/content/FGNET/FGNET/images/074A07.JPG
/content/FGNET/FGNET/images/039A15.JPG
/content/FGNET/FGNET/images/021A23.JPG
/content/FGNET/FGNET/images/046A18.JPG
/content/FGNET/FGNET/images/001A29.JPG
/content/FGNET/FGNET/images/031A19.JPG
/content/FGNET/FGNET/images/047A06.JPG
/content/FGNET/FGNET/images/065A07b.JPG
/content/FGNET/FGNET/images/047A18.JPG
/content/FGNET/FGNET/images/023A21.JPG
/content/FGNET/FGNET/images/080A05.JPG
/content/FGNET/FGNET/

In [ ]:
import os, zipfile, random, shutil, math
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import CosineDecay
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

IMG_SIZE   = 224
BATCH_SIZE = 16


FGNET_ZIP = "/content/FGNET.zip"
MAL_ZIP = None
for cand in ("/content/content.zip", "/content/clean_dataset.zip"):
    if os.path.exists(cand):
        MAL_ZIP = cand
        break
if MAL_ZIP is None:
    raise FileNotFoundError(
        "Malnutrition zip not found. Expected /content/content.zip "
        "or /content/clean_dataset.zip — upload one of those."
    )

with zipfile.ZipFile(FGNET_ZIP, 'r') as z:
    z.extractall("/content/FGNET")
with zipfile.ZipFile(MAL_ZIP, 'r') as z:
    z.extractall("/content/malnutrition_raw")
print(f"Datasets extracted (malnutrition zip: {os.path.basename(MAL_ZIP)})")


def _decode(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return tf.cast(img, tf.float32)

def load_for_densenet(path, label):
    img = _decode(path)
    img = preprocess_input(img)
    return img, label

def augment_face(img_raw, label):
    img = tf.image.random_flip_left_right(img_raw)
    img = tf.image.random_brightness(img, 0.10 * 255.)
    img = tf.image.random_contrast(img, 0.90, 1.10)
    img = tf.clip_by_value(img, 0., 255.)
    crop_size = tf.random.uniform([], int(IMG_SIZE*0.9), IMG_SIZE + 1, dtype=tf.int32)
    img = tf.image.random_crop(img, [crop_size, crop_size, 3])
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = preprocess_input(img)
    return img, label

def load_raw(path, label):
    return _decode(path), label

fgnet_path = "/content/FGNET/FGNET/images"
data = []
for fname in os.listdir(fgnet_path):
    if fname.lower().endswith(".jpg"):
        try:
            age_part = fname.split("A")[1].split(".")[0]
            age = int(''.join(filter(str.isdigit, age_part)))
            data.append([os.path.join(fgnet_path, fname), float(age)])
        except Exception:
            continue

df             = pd.DataFrame(data, columns=["image", "age"])
df["age_norm"] = df["age"] / df["age"].max()
print(f"FG-NET: {len(df)} images | age range {df['age'].min():.0f}-{df['age'].max():.0f}")

fg_pairs = list(zip(df["image"].tolist(), df["age_norm"].tolist()))
random.shuffle(fg_pairs)
cut      = int(0.85 * len(fg_pairs))
fg_train = fg_pairs[:cut]
fg_val   = fg_pairs[cut:]


def make_fg_dataset(pairs, augment):
    paths  = [p for p, _ in pairs]
    labels = [l for _, l in pairs]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.shuffle(len(paths), seed=SEED)
    if augment:
        ds = ds.map(load_raw,      num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.map(augment_face,  num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = ds.map(load_for_densenet, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

fg_train_ds = make_fg_dataset(fg_train, augment=True)
fg_val_ds   = make_fg_dataset(fg_val,   augment=False)
print(f"FG-NET batches — train: {len(fg_train_ds)} | val: {len(fg_val_ds)}")


base_model = DenseNet121(weights='imagenet', include_top=False,
                         input_shape=(IMG_SIZE, IMG_SIZE, 3))

WARMUP_UNFREEZE = 60

for layer in base_model.layers:
    layer.trainable = False
for layer in base_model.layers[-WARMUP_UNFREEZE:]:
    layer.trainable = True

inp = base_model.input
x   = base_model.output
x   = layers.GlobalAveragePooling2D()(x)
x   = layers.Dense(256, activation='relu')(x)
x   = layers.Dropout(0.3)(x)
out = layers.Dense(1, activation='sigmoid')(x)

pretrain_model = models.Model(inp, out)
pretrain_model.compile(optimizer=Adam(1e-4), loss='mse', metrics=['mae'])

print(f"\nFGNET warm-up: {sum(1 for l in pretrain_model.layers if l.trainable)} "
      f"trainable layers (DenseNet121 top-{WARMUP_UNFREEZE})")
print("── Stage 1: FGNET face-age warmup ──")
pretrain_model.fit(
    fg_train_ds, validation_data=fg_val_ds, epochs=12,
    callbacks=[
        callbacks.EarlyStopping(monitor='val_loss', patience=4,
                                restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                    patience=2, min_lr=1e-7, verbose=1)
    ]
)
print("Stage 1 done — DenseNet top-block now face-adapted")

for layer in base_model.layers:
    layer.trainable = False


classes = ["Normal Faces", "abnormal Faces"]

def find_dataset_root(start):
    for dirpath, dirnames, _ in os.walk(start):
        if "__MACOSX" in dirpath:
            continue
        dirnames[:] = [d for d in dirnames if d != "__MACOSX"]
        for sub in dirnames:
            candidate = os.path.join(dirpath, sub)
            if os.path.isdir(os.path.join(candidate, classes[0])) \
               and os.path.isdir(os.path.join(candidate, classes[1])):
                if sub.lower() in ("train", "validation", "val", "test"):
                    return dirpath
                else:
                    return candidate
    return None

src_root = find_dataset_root("/content/malnutrition_raw")
if src_root is None:
    raise FileNotFoundError(
        "Could not locate 'Normal Faces' / 'abnormal Faces' folders inside the zip."
    )
print(f"Dataset root detected: {src_root}")
out_root = "/content/new_dataset"

if os.path.exists(out_root):
    shutil.rmtree(out_root)
for split in ("train", "validation"):
    for cls in classes:
        os.makedirs(os.path.join(out_root, split, cls), exist_ok=True)

def _is_real_image(fname):
    if fname.startswith("."):
        return False
    return fname.lower().endswith(('.jpg', '.jpeg', '.png'))

all_paths, all_labels = [], []
for idx, cls in enumerate(classes):
    found_split = False
    for split in ("train", "validation", "val", "test"):
        cls_dir = os.path.join(src_root, split, cls)
        if os.path.isdir(cls_dir):
            found_split = True
            for f in os.listdir(cls_dir):
                if _is_real_image(f):
                    all_paths.append(os.path.join(cls_dir, f))
                    all_labels.append(idx)
    if not found_split:
        cls_dir = os.path.join(src_root, cls)
        if os.path.isdir(cls_dir):
            for f in os.listdir(cls_dir):
                if _is_real_image(f):
                    all_paths.append(os.path.join(cls_dir, f))
                    all_labels.append(idx)

if len(all_paths) == 0:
    raise RuntimeError(f"No images found under {src_root}.")
print(f"Pooled {len(all_paths)} images from dataset (before stratified split)")

train_p, val_p, train_y, val_y = train_test_split(
    all_paths, all_labels,
    test_size=0.20, stratify=all_labels, random_state=SEED
)

for p, y in zip(train_p, train_y):
    shutil.copy(p, os.path.join(out_root, "train", classes[y]))
for p, y in zip(val_p, val_y):
    shutil.copy(p, os.path.join(out_root, "validation", classes[y]))

print("\nDataset rebuilt (stratified 80/20):")
for split in ("train", "validation"):
    for cls in classes:
        n = len(os.listdir(os.path.join(out_root, split, cls)))
        print(f"  {split:11s} {cls:16s}: {n}")

train_paths  = train_p
train_labels = [float(y) for y in train_y]
val_paths    = val_p
val_labels   = [float(y) for y in val_y]
print(f"Total: train {len(train_paths)} | val {len(val_paths)}")


def make_mal_dataset(paths, labels, augment=False, repeat=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    if augment:
        ds = ds.map(load_raw,      num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.map(augment_face,  num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = ds.map(load_for_densenet, num_parallel_calls=tf.data.AUTOTUNE)
    if repeat:
        ds = ds.repeat()
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_mal_dataset(train_paths, train_labels, augment=True,  repeat=True)
val_ds   = make_mal_dataset(val_paths,   val_labels,   augment=False, repeat=False)

STEPS_PER_EPOCH  = math.ceil(len(train_paths) / BATCH_SIZE)
VALIDATION_STEPS = math.ceil(len(val_paths)   / BATCH_SIZE)
print(f"steps_per_epoch {STEPS_PER_EPOCH} | validation_steps {VALIDATION_STEPS}")

cw = compute_class_weight('balanced',
                           classes=np.array([0, 1]),
                           y=np.array(train_labels))
class_weights = {0: float(cw[0]), 1: float(cw[1])}
print(f"Class weights — Normal {cw[0]:.3f} | Abnormal {cw[1]:.3f}")

x   = base_model.output
x   = layers.GlobalAveragePooling2D()(x)
x   = layers.BatchNormalization()(x)
x   = layers.Dropout(0.3)(x)
x   = layers.Dense(256, activation='relu',
                   kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
x   = layers.BatchNormalization()(x)
x   = layers.Dropout(0.4)(x)
out = layers.Dense(1, activation='sigmoid')(x)

model = models.Model(base_model.input, out)

def metrics():
    return [
        'accuracy',
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
model.compile(
    optimizer=Adam(1e-3),
    loss='binary_crossentropy',
    metrics=metrics()
)

print("\n── Phase A: head-only training (DenseNet121 base frozen, LR 1e-3) ──")
hist_a = model.fit(
    train_ds, validation_data=val_ds, epochs=20,
    steps_per_epoch=STEPS_PER_EPOCH, validation_steps=VALIDATION_STEPS,
    class_weight=class_weights,
    callbacks=[
        callbacks.EarlyStopping(monitor='val_auc', mode='max',
                                patience=8, restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                    patience=3, min_lr=1e-6, verbose=1),
        callbacks.ModelCheckpoint("/content/best_phaseA_densenet.keras",
                                  monitor='val_auc', mode='max',
                                  save_best_only=True, verbose=1),
    ]
)

probs_a = model.predict(
    make_mal_dataset(val_paths, val_labels, augment=False), verbose=0
).flatten()
y_true  = np.array(val_labels).astype(int)
gap_a   = probs_a[y_true==1].mean() - probs_a[y_true==0].mean()
print(f"\nPhase A — score range [{probs_a.min():.3f}, {probs_a.max():.3f}], "
      f"Normal mean {probs_a[y_true==0].mean():.3f}, "
      f"Abnormal mean {probs_a[y_true==1].mean():.3f}, "
      f"gap {gap_a:+.3f}")


FINETUNE_UNFREEZE = 100

for layer in base_model.layers:
    layer.trainable = False
for layer in base_model.layers[-FINETUNE_UNFREEZE:]:
    layer.trainable = True

EPOCHS_B = 30
cosine   = CosineDecay(initial_learning_rate=1e-4,
                       decay_steps=STEPS_PER_EPOCH * EPOCHS_B,
                       alpha=0.01)
model.compile(
    optimizer=Adam(cosine),
    loss='binary_crossentropy',
    metrics=metrics()
)

print(f"\nPhase B — {sum(1 for l in model.layers if l.trainable)} trainable layers "
      f"(DenseNet121 top-{FINETUNE_UNFREEZE})")
print("── Phase B: fine-tune top-100 with cosine LR 1e-4 -> 1e-6 ──")
hist_b = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS_B,
    steps_per_epoch=STEPS_PER_EPOCH, validation_steps=VALIDATION_STEPS,
    class_weight=class_weights,
    callbacks=[
        callbacks.EarlyStopping(monitor='val_auc', mode='max',
                                patience=10, restore_best_weights=True, verbose=1),
        callbacks.ModelCheckpoint("/content/best_final_densenet.keras",
                                  monitor='val_auc', mode='max',
                                  save_best_only=True, verbose=1),
    ]
)


print("\n── Final evaluation with 5-crop TTA ──")

def tta_predict(paths, n_aug=5):
    raw_ds = tf.data.Dataset.from_tensor_slices(paths)
    raw_ds = raw_ds.map(lambda p: _decode(p), num_parallel_calls=tf.data.AUTOTUNE)

    clean_ds = raw_ds.map(lambda i: preprocess_input(i)).batch(BATCH_SIZE)
    probs = model.predict(clean_ds, verbose=0).flatten()

    for _ in range(n_aug - 1):
        def aug(img):
            img = tf.image.random_flip_left_right(img)
            img = tf.image.random_brightness(img, 0.10 * 255.)
            img = tf.clip_by_value(img, 0., 255.)
            return preprocess_input(img)
        aug_ds = raw_ds.map(aug, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE)
        probs += model.predict(aug_ds, verbose=0).flatten()
    return probs / n_aug

probs  = tta_predict(val_paths, n_aug=5)
y_true = np.array(val_labels).astype(int)

gap = probs[y_true==1].mean() - probs[y_true==0].mean()
print(f"TTA score range: [{probs.min():.3f}, {probs.max():.3f}]")
print(f"  Normal mean  : {probs[y_true==0].mean():.3f} (want LOW)")
print(f"  Abnormal mean: {probs[y_true==1].mean():.3f} (want HIGH)")
print(f"  Separation   : {gap:+.3f} (want > 0.15)")

best_thr, best_score = 0.5, 0.0
for t in np.arange(0.20, 0.81, 0.01):
    pred = (probs >= t).astype(int)
    tn   = int(((pred==0) & (y_true==0)).sum())
    tp   = int(((pred==1) & (y_true==1)).sum())
    fp   = int(((pred==1) & (y_true==0)).sum())
    fn   = int(((pred==0) & (y_true==1)).sum())
    sens = tp / max(tp+fn, 1)
    spec = tn / max(tn+fp, 1)
    bal  = 0.5 * (sens + spec)
    if bal > best_score:
        best_score, best_thr = bal, t

pred_default = (probs >= 0.5).astype(int)
pred_tuned   = (probs >= best_thr).astype(int)

def report(pred, tag):
    print(f"\n── {tag} ──")
    print(classification_report(
        y_true, pred, target_names=['Normal', 'Abnormal'],
        digits=3, zero_division=0
    ))
    cm = confusion_matrix(y_true, pred)
    print("Confusion matrix:"); print(cm)
    acc = (pred == y_true).mean()
    print(f"Accuracy: {acc*100:.2f}%")
    return acc

acc_def = report(pred_default, "Threshold 0.50 (default)")
acc_tun = report(pred_tuned,   f"Threshold {best_thr:.2f} (balanced-accuracy tuned)")

model.save("/content/malnutrition_densenet121_v1.keras")
print(f"\nModel saved to /content/malnutrition_densenet121_v1.keras")
print(f"Recommended threshold: {best_thr:.2f}")
print(f"Final accuracy: default {acc_def*100:.2f}% | tuned {acc_tun*100:.2f}%")

Datasets extracted (malnutrition zip: clean_dataset.zip)
FG-NET: 1002 images | age range 0-69
FG-NET batches — train: 54 | val: 10
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

FGNET warm-up: 64 trainable layers (DenseNet121 top-60)
── Stage 1: FGNET face-age warmup ──
Epoch 1/12
54/54 ━━━━━━━━━━━━━━━━━━━━ 99s 993ms/step - loss: 0.0474 - mae: 0.1573 - val_loss: 0.0193 - val_mae: 0.1098 - learning_rate: 1.0000e-04
Epoch 2/12
54/54 ━━━━━━━━━━━━━━━━━━━━ 4s 73ms/step - loss: 0.0237 - mae: 0.1125 - val_loss: 0.0146 - val_mae: 0.0859 - learning_rate: 1.0000e-04
Epoch 3/12
54/54 ━━━━━━━━━━━━━━━━━━━━ 6s 83ms/step - loss: 0.0175 - mae: 0.0970 - val_loss: 0.0142 - val_mae: 0.0910 - learning_rate: 1.0000e-04
Epoch 4/12
54/54 ━━━━━━━━━━━━━━━━━━━━ 5s 73ms/step - loss: 0.0162 - mae: 0.0942 - val_loss: 0.0129 - val_mae: 0.0840 - learning_rate: 1.0000e-04
Epoch 5/12
54/54 ━━━━━━━━━━━━━━━━━━━━ 4s 72ms/step - loss: 0.0129 - mae: 0.0826 - val_loss: 0.0122 - val_mae: 0.0833 - learning_rate: 1.0000e-